<a href="https://colab.research.google.com/github/encen-ya/ML_2026/blob/main/Praktikum6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install faiss-cpu annoy hnswlib

  Using cached faiss_cpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.8 kB)
  Using cached annoy-1.17.3.tar.gz (647 kB)
  Preparing metadata (setup.py) ... done
  Using cached hnswlib-0.8.0.tar.gz (36 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 19.1 MB/s eta 0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520034 sha256=0d0e92882a4a542254f94170edd9ce9f0c48ef9d0769001c70be8ee1cf3995f3
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679797 sha256=3a80c146afa62eee7b8bf83a4b514c3c507c9edefa5a291d9f4b8cb31c918e67
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Succes

In [7]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
from google.colab import files
import io
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------
# Load dataset
# -------------------
uploaded = files.upload()

# Ambil nama file yang baru saja diupload dan load ke DataFrame
file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

if len(df) > 5000:
    df = df.sample(n=5000, random_state=42).reset_index(drop=True)

# Memilih fitur numerik saja sesuai instruksi
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Mengatasi missing values (jika ada) dengan dropna agar tidak error saat scaling
df = df.dropna(subset=features)
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors

print(f"Jumlah data: {X_scaled.shape[0]}, Jumlah fitur: {X_scaled.shape[1]}")

# -------------------
# Exact Nearest Neighbor (brute-force)
# -------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------
# Annoy
# -------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10) # 10 trees
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------
# HNSW
# -------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------
# FAISS IVF
# -------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

Saving songs_with_attributes_and_lyrics.csv to songs_with_attributes_and_lyrics (1).csv
Jumlah data: 5000, Jumlah fitur: 9
Exact NN done in 0.199 s
Annoy done in 0.191 s
HNSW done in 0.672 s
FAISS IVF done in 0.153 s

Top-5 neighbors for first song:
Exact NN: [   0 4665 2045   80 2962]
Annoy:    [0, 4665, 2045, 80, 2962]
HNSW:     [   0 4665 2045   80 2962]
FAISS:    [   0 4665 2045   80 2962]

=== Ringkasan Waktu (detik) ===
Exact NN : 0.199
Annoy    : 0.191
HNSW     : 0.672
FAISS    : 0.153
